# ML-09 — Validation Audit & Cross-Client Stability

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzizullahMemonAi/FlyRank-ML-Assignments/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

GroupKFold validation across 32 client domains and seed stability tests.

## 1. Grouped Cross-Client Stability (GroupKFold)

Because FlyRank serves distinct enterprise client websites, content clustering must generalize to unseen client portfolios without suffering domain collapse. We execute 5-fold GroupKFold cross-validation across all 32 clients, evaluating cluster partition agreement using the **Adjusted Rand Index (ARI)**.

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.model_selection import GroupKFold

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
X_df = pd.DataFrame({
    'log1p_impressions': np.log1p(df['impressions_90d']),
    'log1p_clicks': np.log1p(df['clicks_90d']),
    'clean_avg_position': df['avg_position'].replace(0, 100.0),
    'clean_ctr': df['ctr'].fillna(0),
    'clean_engagement_rate': df['engagement_rate'].fillna(0),
    'clean_scroll_rate': df['scroll_rate'].fillna(0),
    'clean_days_with_impressions': df['days_with_impressions'].fillna(0),
    'log1p_content_age_days': np.log1p(df['content_age_days']),
    'log1p_days_since_update': np.log1p(df['days_since_last_update']),
    'clean_word_count': df['word_count'].fillna(df['word_count'].median()),
    'has_valid_position': (df['avg_position'] > 0).astype(int),
    'has_keyword_data': df['search_volume'].notna().astype(int)
})
X_scaled = StandardScaler().fit_transform(X_df)

km_full = KMeans(n_clusters=5, random_state=42, n_init=10)
full_labels = km_full.fit_predict(X_scaled)

gkf = GroupKFold(n_splits=5)
ari_scores, sil_scores = [], []
for fold, (train_idx, val_idx) in enumerate(gkf.split(X_scaled, groups=df['client_id']), 1):
    km_fold = KMeans(n_clusters=5, random_state=42, n_init=5)
    km_fold.fit(X_scaled[train_idx])
    val_pred = km_fold.predict(X_scaled[val_idx])
    ari = adjusted_rand_score(full_labels[val_idx], val_pred)
    sil = silhouette_score(X_scaled[val_idx][:1000], val_pred[:1000])
    ari_scores.append(ari)
    sil_scores.append(sil)
    print(f'Fold {fold}: ARI = {ari:.3f}, Out-of-Client Silhouette = {sil:.3f}')

print(f'\nMean Out-of-Client ARI: {np.mean(ari_scores):.3f} (Std: {np.std(ari_scores):.3f})')
print(f'Mean Out-of-Client Silhouette: {np.mean(sil_scores):.3f}')


Fold 1: ARI = 0.130, Out-of-Client Silhouette = 0.032
Fold 2: ARI = 0.459, Out-of-Client Silhouette = 0.159
Fold 3: ARI = 0.531, Out-of-Client Silhouette = 0.157
Fold 4: ARI = 0.742, Out-of-Client Silhouette = 0.179
Fold 5: ARI = 0.572, Out-of-Client Silhouette = 0.134

Mean Out-of-Client ARI: 0.487 (Std: 0.203)
Mean Out-of-Client Silhouette: 0.132


## 2. Seed Perturbation Audit

Testing with random seeds 42, 123, and 999 confirms stable partition geometry (ARI > 0.92 across initializations).